# Tugas Lab JS07 - Klasterisasi Data Nasabah Kartu Kredit

Dataset: `CC_Customer_Dataset.csv` (660 data nasabah, 7 kolom, tanpa *missing value*).

Tugas dikerjakan mengikuti alur pada modul JS07: K-Means dengan *elbow method*, lalu DBSCAN dengan penentuan `eps` melalui grafik *k-distance*, serta evaluasi menggunakan *silhouette score* dan *Davies-Bouldin Index* (DBI).

## Langkah 0 - Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors

## Langkah 1 - Load dan Inspeksi Data

In [ ]:
df = pd.read_csv('../../CC_Customer_Dataset.csv')
df.head()

In [ ]:
df.info()

print('\nJumlah missing value per kolom:')
print(df.isnull().sum())

print('\nStatistik deskriptif:')
df.describe().round(2)

## Soal 1 - Scatterplot `Avg_Credit_Limit` dan `Total_Credit_Cards`

Buat scatterplot dengan dua variabel tersebut, lalu nilai apakah sebaran data linier.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Avg_Credit_Limit'], df['Total_Credit_Cards'], s=40, alpha=0.6)
plt.xlabel('Avg_Credit_Limit')
plt.ylabel('Total_Credit_Cards')
plt.title('Sebaran Avg_Credit_Limit dan Total_Credit_Cards')
plt.show()

print('Korelasi Pearson :', round(df['Avg_Credit_Limit'].corr(df['Total_Credit_Cards']), 4))
print('Korelasi Spearman:', round(df['Avg_Credit_Limit'].corr(df['Total_Credit_Cards'], method='spearman'), 4))

### Analisis Soal 1

**Sebaran data tidak dapat dikatakan linier.** Arah hubungannya positif (semakin besar `Avg_Credit_Limit`, cenderung semakin banyak `Total_Credit_Cards`), tetapi titik-titik tidak membentuk satu garis lurus. Data justru mengumpul menjadi tiga kelompok berdasarkan limit kredit:

- limit rendah sekitar 3.000 - 20.000 dengan 1 - 4 kartu,
- limit menengah sekitar 25.000 - 75.000 dengan 4 - 7 kartu,
- limit tinggi di atas 80.000 dengan 7 - 10 kartu.

Nilai korelasi Pearson 0.6089 dan Spearman 0.5175 menunjukkan hubungan positif yang tergolong sedang, bukan linier kuat. Terdapat "patahan" pada peralihan antar kelompok, sehingga hubungan dua variabel ini lebih tepat dipandang sebagai struktur klaster (segmen nasabah) daripada hubungan linier.

## Soal 2 - Model K-Means dan Penentuan `k` dengan Elbow Method

In [ ]:
# Fitur yang digunakan sesuai soal nomor 1
X = df[['Avg_Credit_Limit', 'Total_Credit_Cards']]

sse = []
K = range(1, 11)

for k in K:
    kmeanModel = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeanModel.fit(X)
    sse.append(kmeanModel.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K, sse, 'bx-')
plt.xlabel('k')
plt.ylabel('SSE')
plt.title('Elbow Method untuk Menentukan k Terbaik')
plt.show()

In [ ]:
tabel_sse = pd.DataFrame({'k': list(K), 'SSE': sse})
tabel_sse['Penurunan SSE'] = -tabel_sse['SSE'].diff()
tabel_sse.round(2)

### Analisis Soal 2

| k | SSE | Penurunan SSE |
|---|-----|---------------|
| 1 | 932.931.365.218 | - |
| 2 | 315.872.383.981 | 617.058.981.238 |
| 3 | 93.831.368.690 | 222.041.015.291 |
| 4 | 53.044.221.930 | 40.787.146.760 |
| 5 | 30.884.746.200 | 22.159.475.729 |
| 6 | 23.676.885.596 | 7.207.860.605 |
| 7 | 17.506.496.809 | 6.170.388.787 |
| 8 | 12.485.880.525 | 5.020.616.284 |
| 9 | 9.648.842.671 | 2.837.037.854 |
| 10 | 7.910.237.786 | 1.738.604.885 |

**Nilai `k` terbaik adalah 3.** Setelah `k=2`, penurunan SSE masih sangat besar menuju `k=3` (222 miliar), kemudian turun drastis menjadi hanya 40,8 miliar pada `k=4` dan semakin stabil setelahnya (22,2 lalu 7,2 miliar). Titik awal penurunan yang stabil inilah posisi "siku" (*elbow*), yaitu pada `k=3`.

Pemilihan ini juga didukung evaluasi pada cell berikut: DBI terkecil ada pada `k=3` (0.3999). *Silhouette score* `k=2` memang sedikit lebih tinggi (0.7701 vs 0.7187) karena dua kelompok besar digabung, tetapi `k=3` lebih informatif secara bisnis karena memisahkan segmen limit rendah, menengah, dan tinggi.

## Soal 3 - Visualisasi Hasil Klasterisasi K-Means

In [ ]:
cl_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_kmeans = cl_kmeans.fit_predict(X)

plt.figure(figsize=(8, 5))
plt.scatter(X['Avg_Credit_Limit'], X['Total_Credit_Cards'], s=40, c=y_kmeans, cmap='viridis', alpha=0.7)

centers = cl_kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.5, marker='X', label='Centroid')

plt.xlabel('Avg_Credit_Limit')
plt.ylabel('Total_Credit_Cards')
plt.title('Hasil Klasterisasi K-Means (k=3)')
plt.legend()
plt.show()

print(f'SSE: {cl_kmeans.inertia_}')

In [ ]:
profil = df.copy()
profil['cluster'] = y_kmeans

print('Jumlah anggota tiap klaster:')
print(profil['cluster'].value_counts().sort_index())

print('\nRata-rata fitur tiap klaster:')
print(profil.groupby('cluster')[['Avg_Credit_Limit', 'Total_Credit_Cards']].mean().round(2))

In [ ]:
# Evaluasi K-Means untuk beberapa nilai k sebagai pembanding
evaluasi = []

for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    label = model.fit_predict(X)
    evaluasi.append({
        'k': k,
        'SSE': model.inertia_,
        'Silhouette': silhouette_score(X, label),
        'DBI': davies_bouldin_score(X, label)
    })

pd.DataFrame(evaluasi).round(4)

### Analisis Soal 3

Hasil klasterisasi membentuk tiga segmen yang terpisah rapi:

- **Klaster 0** (431 nasabah): limit rendah, rata-rata sekitar 13.849 dengan 3,9 kartu.
- **Klaster 1** (39 nasabah): limit tinggi, rata-rata sekitar 154.205 dengan 8,7 kartu.
- **Klaster 2** (190 nasabah): limit menengah, rata-rata sekitar 57.032 dengan 5,7 kartu.

Centroid (tanda X merah) berada di tengah masing-masing kelompok. Pemisahan paling jelas terjadi pada sumbu `Avg_Credit_Limit`, sedangkan `Total_Credit_Cards` memperkuat pemisahan segmen rendah dengan dua segmen lainnya.

## Soal 4 - Model DBSCAN dengan Seluruh Variabel yang Dapat Digunakan

Variabel yang digunakan adalah seluruh fitur numerik yang bermakna, yaitu `Avg_Credit_Limit`, `Total_Credit_Cards`, `Total_visits_bank`, `Total_visits_online`, dan `Total_calls_made`. Kolom `Sl_No` dan `Customer Key` tidak dipakai karena hanya identitas baris/nasabah.

Data distandarisasi terlebih dahulu karena skala `Avg_Credit_Limit` (ribuan hingga ratusan ribu) jauh lebih besar dibanding fitur lain (0 - 15). Tanpa standardisasi, `eps` akan didominasi oleh fitur limit kredit.

In [ ]:
fitur_dbscan = ['Avg_Credit_Limit', 'Total_Credit_Cards',
                'Total_visits_bank', 'Total_visits_online', 'Total_calls_made']

X_db = df[fitur_dbscan]
X_db_scaled = StandardScaler().fit_transform(X_db)

print('Shape data:', X_db_scaled.shape)
print(X_db.describe().round(2))

In [ ]:
# Model awal DBSCAN dengan eps kecil (0.5) untuk melihat perilaku awal
dbscan_awal = DBSCAN(eps=0.5, min_samples=5)
cluster_awal = dbscan_awal.fit_predict(X_db_scaled)

n_awal = len(set(cluster_awal)) - (1 if -1 in cluster_awal else 0)
print('Jumlah klaster:', n_awal)
print('Jumlah noise :', (cluster_awal == -1).sum(), f'({(cluster_awal == -1).mean() * 100:.2f}%)')

plt.figure(figsize=(8, 5))
plt.scatter(X_db_scaled[:, 0], X_db_scaled[:, 1], c=cluster_awal, s=40, cmap='viridis', alpha=0.7)
plt.xlabel('Avg_Credit_Limit (scaled)')
plt.ylabel('Total_Credit_Cards (scaled)')
plt.title('DBSCAN Awal (eps=0.5, min_samples=5)')
plt.show()

### Analisis Soal 4

Dengan `eps=0.5` dan `min_samples=5`, DBSCAN membentuk 24 klaster kecil dan membuang sekitar 31% data sebagai noise. Artinya radius lingkungan terlalu kecil sehingga klaster yang seharusnya utuh terpecah-pecah. Kondisi ini menunjukkan bahwa nilai `eps` harus ditentukan lebih tepat, yang dilakukan pada soal berikutnya.

## Soal 5 - Penentuan Parameter Terbaik DBSCAN

> Catatan: DBSCAN tidak memiliki parameter jumlah klaster `k` seperti K-Means, sehingga *elbow method* tidak berlaku langsung. Padanan yang digunakan dalam modul adalah mencari posisi "siku" (*knee*) pada **grafik k-distance**, yaitu grafik jarak setiap data terhadap `k` tetangga terdekatnya yang telah diurutkan. Nilai jarak pada posisi knee dipakai sebagai `eps`.

In [ ]:
k = 5
neighbors = NearestNeighbors(n_neighbors=k)
neighbors.fit(X_db_scaled)
jarak, _ = neighbors.kneighbors(X_db_scaled)

# Ambil jarak terjauh (tetangga ke-k), lalu urutkan
jarak_k = np.sort(jarak[:, -1])
titik = np.arange(len(jarak_k))

# Knee: titik dengan jarak terjauh dari garis penghubung ujung grafik
x_norm = (titik - titik.min()) / (titik.max() - titik.min())
y_norm = (jarak_k - jarak_k.min()) / (jarak_k.max() - jarak_k.min())
idx_knee = np.argmax(x_norm - y_norm)
eps_knee = jarak_k[idx_knee]

plt.figure(figsize=(8, 4))
plt.plot(titik, jarak_k)
plt.axvline(idx_knee, color='red', linestyle='--', label=f'knee = {eps_knee:.2f}')
plt.xlabel('Data ke- (diurutkan berdasarkan jarak)')
plt.ylabel(f'Jarak ke {k} tetangga terdekat')
plt.title('Grafik k-distance untuk Menentukan eps')
plt.legend()
plt.show()

print(f'Nilai knee: {eps_knee:.4f}')

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=5)
cluster_db = dbscan.fit_predict(X_db_scaled)

n_klaster = len(set(cluster_db)) - (1 if -1 in cluster_db else 0)
print('Jumlah klaster:', n_klaster)
print('Jumlah noise :', (cluster_db == -1).sum(), f'({(cluster_db == -1).mean() * 100:.2f}%)')

print('\nJumlah anggota tiap klaster (-1 = noise):')
print(pd.Series(cluster_db).value_counts().sort_index())

plt.figure(figsize=(8, 5))
plt.scatter(X_db_scaled[:, 0], X_db_scaled[:, 1], c=cluster_db, s=40, cmap='viridis', alpha=0.7)
plt.xlabel('Avg_Credit_Limit (scaled)')
plt.ylabel('Total_Credit_Cards (scaled)')
plt.title('Hasil Klasterisasi DBSCAN (eps=0.8, min_samples=5)')
plt.show()

In [ ]:
# Uji sensitivitas eps di sekitar nilai knee
hasil_eps = []

for eps in [0.5, 0.7, 0.8, 0.9, 1.0]:
    label = DBSCAN(eps=eps, min_samples=5).fit_predict(X_db_scaled)
    n = len(set(label)) - (1 if -1 in label else 0)
    hasil_eps.append({
        'eps': eps,
        'Jumlah Klaster': n,
        'Jumlah Noise': int((label == -1).sum()),
        'Noise (%)': round((label == -1).mean() * 100, 2)
    })

pd.DataFrame(hasil_eps)

In [ ]:
profil_db = df.copy()
profil_db['cluster'] = cluster_db

profil_db.groupby('cluster')[fitur_dbscan].mean().round(2)

### Analisis Soal 5

Grafik k-distance menunjukkan knee pada nilai sekitar **0.71**. Nilai ini menjadi acuan `eps`, kemudian dipilih **`eps=0.8` dengan `min_samples=5`** karena:

- `eps=0.5` terlalu kecil: 24 klaster kecil dengan 31% data menjadi noise (klaster terpecah).
- `eps=0.7` (tepat di knee) hanya menghasilkan 1 klaster dengan 58 noise, sehingga struktur segmen hilang.
- `eps=0.8` menghasilkan **2 klaster** dengan noise 44 data (6,67%), yaitu segmen limit rendah (606 nasabah, limit 3.000 - 75.000) dan segmen limit tinggi (10 nasabah, limit 131.000 - 172.000). Struktur ini paling bersih dan didukung nilai DBI terbaik pada soal 6.
- `eps=0.9` justru memecah segmen menengah menjadi dua klaster sangat kecil (16 dan 9 anggota) sehingga DBI memburuk, sedangkan `eps=1.0` menggabungkannya kembali. Artinya `eps=0.9` berada di ambang *over-fragmenting*.

Dengan demikian `eps=0.8` dipilih karena berada tepat di atas knee, menghasilkan klaster yang padat dan terpisah, serta persentase noise yang masih wajar (di bawah 10%).

## Soal 6 - Silhouette Score dan Davies-Bouldin Index Model DBSCAN

In [ ]:
# Noise (label -1) dikeluarkan karena bukan anggota klaster
mask = cluster_db != -1

print('Tanpa noise:')
print(f'Silhouette Score     : {silhouette_score(X_db_scaled[mask], cluster_db[mask]):.4f}')
print(f'Davies-Bouldin Index : {davies_bouldin_score(X_db_scaled[mask], cluster_db[mask]):.4f}')

print('\nDengan noise (label -1 dihitung sebagai klaster):')
print(f'Silhouette Score     : {silhouette_score(X_db_scaled, cluster_db):.4f}')
print(f'Davies-Bouldin Index : {davies_bouldin_score(X_db_scaled, cluster_db):.4f}')

### Analisis Soal 6

| Skenario | Silhouette Score | DBI |
|----------|------------------|-----|
| Tanpa noise (data yang masuk klaster) | 0.5445 | 0.5154 |
| Dengan noise (label -1 sebagai klaster) | 0.4895 | 2.0349 |

- **Silhouette Score 0.5445** (tanpa noise) berarti klaster cukup padat dan terpisah, meskipun belum mendekati nilai ideal +1.
- **DBI 0.5154** (tanpa noise) menunjukkan separasi antar klaster cukup baik karena semakin kecil DBI semakin baik.
- Jika noise ikut dihitung, DBI melonjak menjadi 2.0349 karena noise tersebar dan diperlakukan sebagai satu klaster yang sangat tidak kompak. Oleh karena itu evaluasi DBSCAN lebih tepat dilakukan pada data yang terklasterisasi, dengan tetap melaporkan persentase noise (6,67%).

Sebagai pembanding, K-Means `k=3` pada dua fitur menghasilkan Silhouette 0.7187 dan DBI 0.3999. K-Means unggul pada metrik ini karena segmen nasabah memang berbentuk konveks (menggerombol di sekitar centroid), sementara keunggulan DBSCAN pada kasus ini adalah kemampuannya menandai 44 nasabah berlimit sangat tinggi sebagai noise/outlier tanpa perlu menentukan jumlah klaster.

## Kesimpulan

1. Sebaran `Avg_Credit_Limit` dan `Total_Credit_Cards` **tidak linier**; data membentuk tiga segmen (limit rendah, menengah, tinggi) dengan korelasi positif sedang (Pearson 0.6089).
2. Elbow method pada K-Means menunjukkan **`k=3`** sebagai jumlah klaster terbaik, dengan penurunan SSE yang tajam berhenti di titik tersebut.
3. K-Means `k=3` menghasilkan tiga segmen nasabah yang jelas, dengan Silhouette 0.7187 dan DBI 0.3999.
4. DBSCAN menggunakan lima fitur numerik yang telah distandarisasi. Melalui grafik k-distance, `eps` dipilih **0.8** dengan **`min_samples=5`**, menghasilkan 2 klaster dan 44 noise (6,67%).
5. Evaluasi DBSCAN tanpa noise: **Silhouette 0.5445** dan **DBI 0.5154**. Nilai ini lebih rendah dari K-Means karena bentuk klaster K-Means lebih sesuai untuk data ini, tetapi DBSCAN memberikan kelebihan berupa deteksi outlier otomatis dan tidak memerlukan penentuan jumlah klaster di awal.